# The Beatles, Part A: Clean and Tidy Data, by Example

**Beatles notebooks:** **A. Clean & Tidy Examples** · [B. Clean Data](B_Beatles_Clean_Data.ipynb) · [C. Tidy Data](C_Beatles_Tidy_Data.ipynb) · [D. Find, Filter, Group](D_Beatles_Find_Filter_Group.ipynb) · [E. Charts and Graphs](E_Beatles_Charts_Graphs.ipynb)

**The big picture.** Real data is almost never ready to use. Before we can filter, group, or chart it, we have to fix the ways in which it doesn't say what we mean. This notebook shows **four typical problems**, each with one example from our Beatles data:

**Clean data**
1. **Missing values** that actually carry information (Billboard Top 50)
2. **Inconsistent values**: the same thing spelled different ways (album names)

**Tidy data**

3. **Two variables in one column** (UK and US album debuts)
4. **Several observations in one cell** (genre tags)

Each fix is a **decision** about what the data means. That is why cleaning is part of the research, not a chore before it.

## 1. Load the Data

`Beatles_Belgrade_Complete.csv` combines **Spotify** audio measurements with metadata compiled at the **University of Belgrade**: songwriters, albums, genre labels, chart histories, and more.

In [24]:
import pandas as pd

beatles_raw = pd.read_csv('Beatles_Belgrade_Complete.csv')
beatles_raw.shape

(285, 45)

The file has 45 columns. For our examples we only need a few, and we give them short names that are easy to type. `rename()` takes a **dictionary**: old name → new name.

In [25]:
column_names = {
    'Title': 'title',
    'Year': 'year',
    'Album': 'album',
    'Album debut': 'album_debut',
    'Genre': 'genre',
    'Top 50 (Billboard)': 'top_50_billboard',
}

# always work on a copy, so the raw data stays untouched
beatles = beatles_raw.rename(columns=column_names)[list(column_names.values())].copy()
beatles.head()

,title,year,album,album_debut,genre,top_50_billboard
0,(You're So Square) Baby I Don't Care - Studio Jam,NaN,The Beatles,NaN,NaN,NaN
1,12-bar Original,1965,Anthology 2,Anthology 2,Blues,NaN
2,A Beginning - Anthology 3 Version,NaN,Anthology 3,NaN,NaN,NaN
3,A Day in the Life,1967,Sgt. Pepper's Lonely Hearts Club Band,Sgt. Pepper's Lonely Hearts Club Band,"Psychedelic Rock, Art Rock, Pop/Rock",NaN
4,A Hard Day's Night,1964,A Hard Day's Night,UK: A Hard Day's Night US: 1962-1966,"Rock, Electronic, Pop/Rock",8.0


---
# Part I: Clean Data

## 2. Missing Values: What Does an Empty Cell Mean?

`isna().sum()` counts the missing values (`NaN`) in each column:

In [26]:
beatles.isna().sum()

title                 0
year                  8
album                 1
album_debut          10
genre                 7
top_50_billboard    236
dtype: int64

`top_50_billboard` is empty for almost every song. Why? The column holds each song's **rank** in Billboard's list of the 50 greatest Beatles songs (1 = best). Most songs simply aren't on that list.

So here `NaN` is not an error. It is **information**: "not in the Billboard Top 50."

What should we do with it?

* **Don't fill it with `0`.** A rank of 0 would look like "better than No. 1"!
* **Don't drop these rows.** We'd lose most of our songs.
* **Keep the ranks**, and add a simple **True/False column** that says what the empty cells mean. `.notna()` returns `True` wherever there *is* a value.  So `True` means "in the Top 50," and `False` means "not in the Top 50."

In [27]:
beatles['billboard_top50_hit'] = beatles['top_50_billboard'].notna()

beatles[['title', 'top_50_billboard', 'billboard_top50_hit']].head(10)

,title,top_50_billboard,billboard_top50_hit
0,(You're So Square) Baby I Don't Care - Studio Jam,NaN,False
1,12-bar Original,NaN,False
2,A Beginning - Anthology 3 Version,NaN,False
3,A Day in the Life,NaN,False
4,A Hard Day's Night,8.0,True
5,A Shot of Rhythm and Blues,NaN,False
6,A Taste of Honey,NaN,False
7,Across the Universe,NaN,False
8,Across the Universe - Take 2 / Anthology 2 Ver...,NaN,False
9,Act Naturally,50.0,True


Now the question "how many songs made the list?" is easy to answer:

In [28]:
beatles['billboard_top50_hit'].value_counts()

billboard_top50_hit
False    236
True      49
Name: count, dtype: int64

> **Not all missing data means the same thing.**  For example, elsewhere in this file, a missing `lead_vocal` probably means "instrumental," and a missing `genre` means the song was never labeled. Always ask *why* a value is missing before deciding to fill, drop, or keep it.

## 3. Inconsistent Values: `replace()`

A quick way to spot inconsistencies is to look at every **unique value** with its count:

In [29]:
beatles['album'].value_counts()

album
The Beatles                                34
Live at the BBC                            27
Past Masters (Vols. 1 & 2 / Remastered)    26
Anthology 1                                21
Abbey Road                                 17
Beatles for Sale                           14
Revolver                                   14
With the Beatles                           14
Help!                                      14
Please Please Me                           14
Rubber Soul                                14
A Hard Day's Night                         13
Sgt. Pepper's Lonely Hearts Club Band      13
Let It Be                                  12
Magical Mystery Tour                       11
Anthology 3                                10
Anthology 2                                 5
Yellow Submarine                            4
Live At The BBC                             3
On Air – Live at the BBC Volume 2           1
Abbey Road (Super Deluxe Edition)           1
On Air - Live At The BBC (Vo

Look closely at the bottom of the list:

* `Live at the BBC` and `Live At The BBC` (capitalization not the same!)
* `On Air – Live at the BBC Volume 2` and `On Air - Live At The BBC (Vol.2)` (different dashes and wording)
* `Abbey Road (Super Deluxe Edition)` is a reissue of `Abbey Road`.  Do they have the same tracks?  Do we want to count them separately, or as one album?


We need to change **just certain values** leaving all the others alone. That's a job for `replace()`. Its dictionary's **keys** are the old values and its **values** are the new ones:

In [30]:
# the dictionary uses the OLD titles as the keys and the NEW titles as the values.  


album_fixes = {
    'Live At The BBC': 'Live at the BBC',
    'On Air - Live At The BBC (Vol.2)': 'On Air – Live at the BBC Volume 2',
    'Abbey Road (Super Deluxe Edition)': 'Abbey Road',
}



# The `replace()` method will look for the keys and replace them with the values.
beatles['album'] = beatles['album'].replace(album_fixes)

# and how we can use `value_counts()` to check the results.
beatles['album'].value_counts()

album
The Beatles                                34
Live at the BBC                            30
Past Masters (Vols. 1 & 2 / Remastered)    26
Anthology 1                                21
Abbey Road                                 18
Rubber Soul                                14
Beatles for Sale                           14
Revolver                                   14
With the Beatles                           14
Help!                                      14
Please Please Me                           14
A Hard Day's Night                         13
Sgt. Pepper's Lonely Hearts Club Band      13
Let It Be                                  12
Magical Mystery Tour                       11
Anthology 3                                10
Anthology 2                                 5
Yellow Submarine                            4
On Air – Live at the BBC Volume 2           2
Rock 'n' Roll Music                         1
Name: count, dtype: int64

`Live at the BBC` now has 30 songs (27 + 3), and `Abbey Road` has 18 (17 + 1).

> `replace()` only touches the values listed in the dictionary. That's safe, but it also means *you* have to find every problem. Checking `value_counts()` before and after is a good habit.

---
# Part II: Tidy Data

**Tidy data** follows a few simple principles (from Hadley Wickham's article ["Tidy Data"](https://www.jstatsoft.org/article/view/v059i10)):

1. each **variable** forms a column
2. each **observation** forms a row

Our table breaks both rules. Let's look at the two problem columns and how to fix them:

* `album_debut` sometimes holds **two variables** in one cell: the UK album *and* the US album.
* `genre` holds **several observations** in one cell: `Psychedelic Rock, Art Rock, Pop/Rock` means "this song is Psychedelic Rock," "this song is Art Rock," and "this song is Pop/Rock."

In [31]:
beatles.loc[[3, 4, 6], ['title', 'album_debut', 'genre']]

,title,album_debut,genre
3,A Day in the Life,Sgt. Pepper's Lonely Hearts Club Band,"Psychedelic Rock, Art Rock, Pop/Rock"
4,A Hard Day's Night,UK: A Hard Day's Night US: 1962-1966,"Rock, Electronic, Pop/Rock"
6,A Taste of Honey,UK: Please Please Me US: The Early Beatles,"Pop/Rock, Jazz, Stage&Screen"


## 4. Tidy Data Problem 1:  Two Albums in One Column



In the 1960s, the Beatles' records were packaged differently in Britain and America, so many songs first appeared on one album in the UK and a *different* album in the US.

The pattern is always `UK: ... US: ...`. 

We can thus use the Pythong `split()` method to to break the string into two pieces.  The first piece is the UK album, and the second piece is the US album.

Run the cells below to see how this method works on a single sample string:



In [32]:
sample_combined_string = "UK: A Hard Day's Night US: 1962-1966"

# split at ' US: ' to get a list of two pieces
parts = sample_combined_string.split(' US: ')
parts




["UK: A Hard Day's Night", '1962-1966']

In [33]:
# the result of the `split()` is a `list`; we can get each item in the list separately with parts[0] and parts[1]

# ad the same time we can use `removeprefix()` to remove the leading "UK: " from the first item in the list, since we no longer want that!

uk_album = parts[0].removeprefix('UK: ')
us_album = parts[1]

uk_album, us_album


("A Hard Day's Night", '1962-1966')

In [34]:
# Now that we know how to split a single string, we can use the same method to split the entire `album_debut` column into two new columns: `album_debut_uk` and `album_debut_us`.

# we operate on the ENTIRE COLUMN at once:  `beatles['album_debut'].str.split(' US: ', expand=True)`
# and we assign the result to two new columns: `beatles[['album_debut_uk', 'album_debut_us']]`
beatles[['album_debut_uk', 'album_debut_us']] = beatles['album_debut'].str.split(' US: ', expand=True)

# here we can compare the original `album_debut` column with the two new columns to see that the split worked correctly.
beatles.loc[[3, 4, 6], ['title', 'album_debut', 'album_debut_uk', 'album_debut_us']]

,title,album_debut,album_debut_uk,album_debut_us
3,A Day in the Life,Sgt. Pepper's Lonely Hearts Club Band,Sgt. Pepper's Lonely Hearts Club Band,None
4,A Hard Day's Night,UK: A Hard Day's Night US: 1962-1966,UK: A Hard Day's Night,1962-1966
6,A Taste of Honey,UK: Please Please Me US: The Early Beatles,UK: Please Please Me,The Early Beatles


One small problem remains:

1. Where there was only one album (the same in both countries), the US column is empty. `fillna()` can fill each empty cell with the value **from another column in the same row**.

In [35]:
beatles['album_debut_us'] = beatles['album_debut_us'].fillna(beatles['album_debut_uk'])

beatles.loc[[3, 4, 6], ['title', 'album_debut_uk', 'album_debut_us']]

,title,album_debut_uk,album_debut_us
3,A Day in the Life,Sgt. Pepper's Lonely Hearts Club Band,Sgt. Pepper's Lonely Hearts Club Band
4,A Hard Day's Night,UK: A Hard Day's Night,1962-1966
6,A Taste of Honey,UK: Please Please Me,The Early Beatles


Now the UK and US releases are separate variables, and a question like "which US albums did songs from the UK *Help!* end up on?" takes one line!

We will learn more about this kind of conditional filtering in the next notebook, [D. Find, Filter, Group](D_Beatles_Find_Filter_Group.ipynb).

In [36]:
beatles[beatles['album_debut_uk'] == 'Help!']['album_debut_us'].value_counts()

album_debut_us
Help!    7
Name: count, dtype: int64

## 5. Tidy Data Problem 2:  Multiple Genres as a Single String

To Python, `'Psychedelic Rock, Art Rock, Pop/Rock'` is one long piece of text, not three genres. We can't count genres or ask "which songs are Art Rock?" until each label stands on its own. This takes two steps.

First, one more kind of missing data. Seven tracks (studio jams and outtakes that exist only in the Spotify data) have *no genre at all*. They can't take part in a genre analysis, so we drop them. `dropna(subset=['genre'])` drops rows that are missing a value *in that column only*:

In [37]:
beatles = beatles.dropna(subset=['genre'])
beatles.shape

(278, 9)

### Step 1: turn each string into a list

Here we use the `str.split()` method again, which operates on a **pandas Series** (a single column of a DataFrame) at once, just as we did for the album debut column above.

`str.split(', ')` cuts the text at every comma-and-space and returns a Python **list**:

In [47]:
# here we split the old `genre` column into a new column called `genre_split`, which contains a **list** of genres for each song.

beatles['genre_split'] = beatles['genre'].str.split(', ')

# let's look at the results for a few songs that have multiple genres, to see that the split worked correctly.

beatles.loc[[3, 4, 6], ['title', 'genre', 'genre_split']]

,title,genre,genre_split
3,A Day in the Life,"Psychedelic Rock, Art Rock, Pop/Rock","[Psychedelic Rock, Art Rock, Pop/Rock]"
4,A Hard Day's Night,"Rock, Electronic, Pop/Rock","[Rock, Electronic, Pop/Rock]"
6,A Taste of Honey,"Pop/Rock, Jazz, Stage&Screen","[Pop/Rock, Jazz, Stage&Screen]"


The square brackets show that each cell now holds a list. For example, `.str.len()` counts the labels in each list:

In [49]:
# how MANY genre tags per song?

beatles['genre_split'].str.len()

1      1
3      3
4      3
5      2
6      3
      ..
280    2
281    1
282    2
283    1
284    4
Name: genre_split, Length: 278, dtype: int64

In [52]:
# you could even see the distribution of the number of genre tags per song with `value_counts()`


beatles['genre_split'].str.len().value_counts().sort_index()

genre_split
1     54
2    141
3     61
4     17
5      4
6      1
Name: count, dtype: int64

### Step 2: `explode()` the lists

Having lists of genres in a single column is fairly legible for human readers working with a small dataset. But for analysis, we need each genre to have its own row.  This way we can easily find (for example) all the songs that are Art Rock, or count how many songs are Psychedelic Rock.  Or we could could group data by genre and album, or genre in relation to some Spotify variable like `energy` or `danceability`.

The key here is to have **one observation per row**.  This is a key principle of **tidy data**: each row is one observation, and each column is one variable.

There is a simple way to do this in pandas: `explode()`.

`explode('genre')` gives every item in each list **its own row**. All the other columns are copied into each new row. The new rows keep their old index numbers, so we reset the index.

The result at first seems strange: the same song appears in multiple rows. But now each row is one observation, and each column is one variable.  This is **tidy data**!

In [56]:
# exploding the original dataframe to create a new dataframe with one row per genre tag
# note that the original beatles dataframe is NOT changed!  

beatles_tidy = beatles.explode('genre_split').reset_index(drop=True)


# the resulting tidy dataframe is now MUCH longer than the original, because each song that had multiple genres now has multiple rows, one for each genre.

print('rows before:', len(beatles))
print('rows after: ', len(beatles_tidy))

rows before: 278
rows after:  613


Here is "A Day in the Life" before and after:

In [57]:
beatles[beatles['title'] == 'A Day in the Life'][['title', 'year', 'genre']]

,title,year,genre
3,A Day in the Life,1967,"Psychedelic Rock, Art Rock, Pop/Rock"


In [58]:
beatles_tidy[beatles_tidy['title'] == 'A Day in the Life'][['title', 'year', 'genre']]

,title,year,genre
1,A Day in the Life,1967,"Psychedelic Rock, Art Rock, Pop/Rock"
2,A Day in the Life,1967,"Psychedelic Rock, Art Rock, Pop/Rock"
3,A Day in the Life,1967,"Psychedelic Rock, Art Rock, Pop/Rock"


Now we can count individual genres directly:

In [60]:
beatles_tidy['genre_split'].value_counts().head(10)

genre_split
Pop/Rock            273
Rock and Roll        45
Rock                 39
Psychedelic Rock     28
Folk Rock            21
Hard Rock            18
R&B                  16
Blues Rock           11
Merseybeat           10
Baroque Pop          10
Name: count, dtype: int64

> **Caution: one row is no longer one song.** A song with three genres now has three rows. Use the tidy (exploded) table for questions about **labels** ("How many songs are Folk Rock?"), and the one-row-per-song table for questions about **songs** ("How many songs did Lennon write?").

---
## Summary

**Clean data**
* **Missing values that mean something:** keep them, and add a True/False column with `.notna()`
* **Inconsistent values:** fix specific values with `.replace(dictionary)`

**Tidy data**
* **Two variables in one column:** `.str.split(..., expand=True)`, then `.str.removeprefix()` and `.fillna(other_column)`
* **Several observations in one cell:** `.str.split()` into lists, then `.explode()`

> **Reproducibility:** it is tempting to fix errors by hand in the spreadsheet. But if every change is made in code, anyone can start from the original CSV, get exactly your result, and see every decision you made along the way.

**Next:** in **Notebook B** we look at cleaning in more depth, and in **Notebook C** at tidying.

**Beatles notebooks:** **A. Clean & Tidy Examples** · [B. Clean Data](B_Beatles_Clean_Data.ipynb) · [C. Tidy Data](C_Beatles_Tidy_Data.ipynb) · [D. Find, Filter, Group](D_Beatles_Find_Filter_Group.ipynb) · [E. Charts and Graphs](E_Beatles_Charts_Graphs.ipynb)